# AI1 — So sánh M1 cũ / Arm C trên CV cá nhân

**Riêng tư:** notebook chứa các đoạn kinh nghiệm/dự án từ CV (không có header liên hệ), không phải dữ liệu công khai. Giữ notebook và dataset ở chế độ Private. Không chia sẻ output công khai.

Đây là demo retrieval trên một CV và JD cũ; không phải test độc lập hoặc kết luận chất lượng tổng quát. Pool chỉ gồm 20 bullet, không phải toàn bộ CV. Không có nhãn gold nên không tính accuracy, không chọn model bằng cosine cao hơn.

Notebook KHÔNG train, KHÔNG cài package, KHÔNG thay runtime ứng dụng. Model và tokenizer được nạp offline từ safetensors. Chạy trong phiên training còn checkpoint, hoặc Add Input các model đã lưu (dạng thư mục đã giải nén). Nếu chỉ còn ZIP, cần giải nén an toàn riêng trước.


In [ ]:
from pathlib import Path
import hashlib, json, sys, subprocess, tempfile

# Để None để tự tìm đúng weights theo checksum. Nếu tìm không thấy,
# điền đường dẫn THƯ MỤC model dưới /kaggle/input hoặc /kaggle/working.
BASELINE_DIR = None
ARM_C_DIR = None
EXPECTED = {
    'baseline': '871aeef8a305b0336b443ce8ed1c275e0a656eee6760d87d8cab1b6665c829df',
    'arm_c': '3bdc4ec9783e35a03b3c22c036a24edd871b3be5389aa42f62c906a20b885920',
}

def sha256(p):
    with p.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

candidates = []
for location in ['/kaggle/input', '/kaggle/working']:
    base = Path(location)
    if base.exists():
        candidates.extend(base.rglob('model.safetensors'))

def locate(configured, expected):
    paths = [Path(configured) / 'model.safetensors'] if configured else candidates
    for weight in paths:
        if weight.is_symlink() or not weight.is_file():
            continue
        if sha256(weight) == expected:
            folder = weight.parent.resolve()
            if not (folder / 'config.json').is_file():
                raise RuntimeError(f'Thiếu config: {folder}')
            print('Đã xác minh model:', folder)
            return folder
    raise FileNotFoundError(
        'Chưa tìm thấy model đúng checksum. Add Input các thư mục model đã lưu, '
        'hoặc chạy notebook này trong phiên còn checkpoint. Không cần train lại '
        'nếu bạn còn ZIP model trên máy.'
    )

baseline = locate(BASELINE_DIR, EXPECTED['baseline'])
arm_c = locate(ARM_C_DIR, EXPECTED['arm_c'])
work = Path(tempfile.mkdtemp(prefix='ai1_cv_demo_', dir='/kaggle/working'))
print('Thư mục chạy mới:', work)


## Đầu vào cố định trước inference

Requirement là bản diễn đạt từ JD Sora Fullstack trong raw lịch sử; chưa xác minh JD không trùng training. Evidence trích bullet bằng pypdf, chuẩn hóa NFKC; có thể còn lỗi layout như `T ypeScript`. Không tự sửa nhãn sau khi xem điểm. Đây là so sánh retrieval không có ngưỡng trạng thái, không suy ra ứng viên đáp ứng từ cosine.


In [ ]:
INPUT = {'purpose': 'Exploratory retrieval-only demo; no human-adjudicated support labels or accuracy claims', 'source_cv_sha256': 'a1cc534ba59eb5a235ae3a28c89258b802aaf1f81d8755fdf5d1534dd14f9fbf', 'extraction': 'Local pypdf; NFKC normalization; bullet passages only, not entire CV; contact header excluded', 'limitations': ['Bullet-only pool excludes summary, skills lists, education and certification.', 'PDF extraction retains layout artifacts; passages require manual review.', 'JD from historical raw dataset; training overlap unknown.', 'No per-evidence gold labels.'], 'job_id': 'itviec_mid-fullstack-dev-python-java-nodejs-react-vue-sora-technologies-2723', 'requirements': [{'id': 'R1', 'text': 'Phát triển backend (REST API, business logic, tích hợp dịch vụ) bằng Python, Node.js, Java hoặc C#.'}, {'id': 'R2', 'text': 'Xây dựng frontend SPA bằng React hoặc Vue.'}, {'id': 'R3', 'text': 'Viết và bảo trì script tự động hoá, ETL, data processing hoặc service phụ trợ.'}, {'id': 'R4', 'text': 'Ứng dụng công cụ AI coding assistant trong phát triển phần mềm để tăng năng suất và chất lượng code.'}, {'id': 'R5', 'text': 'Tham gia review code, đóng góp quy trình kỹ thuật và mentor junior khi cần.'}, {'id': 'R6', 'text': 'Làm việc trực tiếp với Tech Lead, BA và team Nhật theo quy trình Horenso: chủ động báo cáo, liên lạc, tham vấn.'}], 'evidence': [{'id': 'E01', 'page': 1, 'section': 'EXPERIENCE', 'text': 'Contributed to two products, Lumina Core and Lumina Plus (see Projects), across frontend and backend in an Agile team (sprint planning, code reviews)'}, {'id': 'E02', 'page': 1, 'section': 'EXPERIENCE', 'text': 'Developed CRUD APIs with FastAPI and Alembic migrations; added test coverage checks to GitLab CI'}, {'id': 'E03', 'page': 1, 'section': 'EXPERIENCE', 'text': 'Adopted an AI-integrated workflow (Claude in GitLab) for spec/plan-before-code and automated code review; contributed to an AI assistant feature and LLM client design'}, {'id': 'E04', 'page': 1, 'section': 'PROJECTS', 'text': 'Tech: React 18, T ypeScript, Vite, T anStack Query, T ailwind CSS 4, Radix UI, Recharts, i18next, Azure MSAL, Vitest, Playwright, FastAPI, SQLAlchemy, Alembic, PostgreSQL, Redis, MinIO, Docker, GitLab CI'}, {'id': 'E05', 'page': 1, 'section': 'PROJECTS', 'text': 'Built Dashboard Snapshot PDF export end-to-end: preview UI with i18n (vi/en/ja) and a backend render/storage pipeline streaming files directly to the client'}, {'id': 'E06', 'page': 1, 'section': 'PROJECTS', 'text': 'Implemented a DB-driven onboarding tour with Vitest and Playwright e2e tests, and sidebar app pinning with touch-friendly drag-reorder and offline UX'}, {'id': 'E07', 'page': 1, 'section': 'PROJECTS', 'text': 'Implemented granular RBAC (role, department, resource level), app-store/workplace admin pages and AI-model-to-app assignment'}, {'id': 'E08', 'page': 1, 'section': 'PROJECTS', 'text': 'Tech: T auri v2 + Rust, React 19, T ypeScript, Mantine, React Flow, Python (LangGraph, Playwright), FastAPI, WebSocket, SQLAlchemy, PostgreSQL'}, {'id': 'E09', 'page': 1, 'section': 'PROJECTS', 'text': 'Built Zalo OA integration : webhook receiver (signature verification, dedup, WebSocket relay), send/read-message nodes, auto token refresh, multi-account'}, {'id': 'E10', 'page': 1, 'section': 'PROJECTS', 'text': 'Built workflow templates (API + frontend pages, publish workflow as template), email/SMTP nodes and ZIP extraction node'}, {'id': 'E11', 'page': 1, 'section': 'PROJECTS', 'text': 'Redesigned execution logs as a run tree; added dark mode, remote pairing (link + QR) and Ubuntu .deb packaging'}, {'id': 'E12', 'page': 1, 'section': 'PROJECTS', 'text': 'Worked with the AI agent/copilot engine (LangGraph, multi-LLM, stdio JSON-RPC) and Playwright-based RPA nodes powering the workflow canvas'}, {'id': 'E13', 'page': 1, 'section': 'PROJECTS', 'text': 'Role: Fullstack Developer – solely responsible for frontend and backend'}, {'id': 'E14', 'page': 1, 'section': 'PROJECTS', 'text': 'Tech: React 18, T ypeScript, shadcn/ui, T anStack Query, Zustand, Node.js, Express.js, Prisma, PostgreSQL, Docker'}, {'id': 'E15', 'page': 1, 'section': 'PROJECTS', 'text': 'Designed habit-to-goal reward engine : task completion auto-accumulates reward into financial goals with instant optimistic UI preview ; nightly cron scheduler settles verified transactions to ledger'}, {'id': 'E16', 'page': 1, 'section': 'PROJECTS', 'text': 'Built immutable SHA-256 hash-chain ledger ; Notion OAuth with AI task suggestions via SSE; AI provider abstraction layer (Gemini/Groq/Anthropic) switchable via .env'}, {'id': 'E17', 'page': 2, 'section': '', 'text': 'Role: Frontend Developer – solely responsible for entire client-side across 2 user roles (User, Admin)'}, {'id': 'E18', 'page': 2, 'section': '', 'text': 'Tech: React 19, T ypeScript, Vite, T ailwind CSS 4, Socket.IO, WebRTC, Google OAuth 2.0'}, {'id': 'E19', 'page': 2, 'section': '', 'text': 'Implemented WebRTC P2P video call from scratch (no library) – ICE/SDP negotiation, remote stream, PiP , media controls, and real-time chat alongside video'}, {'id': 'E20', 'page': 2, 'section': '', 'text': 'Engineered Axios interceptor (JWT silent refresh & request queuing) and Socket Singleton; diagnosed & fixed missing end-call signal and duplicate match race condition with backend team'}]}
RUNNER = '"""Offline, label-free CV evidence comparison; never updates model weights."""\n\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport time\n\n\ndef digest(path):\n    with path.open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef run(args):\n    os.environ[\'HF_HUB_OFFLINE\'] = \'1\'\n    os.environ[\'TRANSFORMERS_OFFLINE\'] = \'1\'\n    import torch\n    import transformers\n    from transformers import AutoModel, AutoTokenizer\n\n    data = json.loads(args.input.read_text(encoding=\'utf-8\'))\n    requirements, evidence = data[\'requirements\'], data[\'evidence\']\n    if not requirements or not evidence:\n        raise ValueError(\'Nonempty requirements and evidence required\')\n    if len({e[\'id\'] for e in evidence}) != len(evidence):\n        raise ValueError(\'Duplicate evidence IDs\')\n    texts = [r[\'text\'] for r in requirements] + [e[\'text\'] for e in evidence]\n    result = {\n        \'purpose\': \'Single-CV exploratory retrieval demo, not a heldout benchmark\',\n        \'input_sha256\': digest(args.input),\n        \'environment\': {\'python\': platform.python_version(), \'torch\': torch.__version__,\n                        \'transformers\': transformers.__version__, \'device\': \'cpu\'},\n        \'protocol\': {\'max_tokens\': 256, \'pooling\': \'attention-mask mean\',\n                     \'normalization\': \'L2\', \'top_k\': 3,\n                     \'latency_note\': \'One local encoding pass including tokenization; not a production SLA\'},\n        \'arms\': {},\n    }\n    for arm, folder in [(\'baseline\', args.baseline), (\'arm_c\', args.arm_c)]:\n        folder = folder.resolve()\n        weights = sorted(folder.glob(\'*.safetensors\'))\n        if not weights:\n            raise ValueError(f\'Missing safetensors weights: {folder}\')\n        tokenizer = AutoTokenizer.from_pretrained(\n            str(folder), local_files_only=True, trust_remote_code=False)\n        model = AutoModel.from_pretrained(\n            str(folder), local_files_only=True, trust_remote_code=False,\n            use_safetensors=True).to(\'cpu\').eval()\n        model.requires_grad_(False)\n        vectors, token_lengths = [], []\n        started = time.perf_counter()\n        with torch.inference_mode():\n            for start in range(0, len(texts), 8):\n                chunk = texts[start:start + 8]\n                token_lengths.extend(len(ids) for ids in tokenizer(chunk, truncation=False)[\'input_ids\'])\n                batch = tokenizer(chunk, padding=True, truncation=True,\n                                  max_length=256, return_tensors=\'pt\')\n                hidden = model(input_ids=batch[\'input_ids\'],\n                               attention_mask=batch[\'attention_mask\']).last_hidden_state\n                mask = batch[\'attention_mask\'].unsqueeze(-1).to(hidden.dtype)\n                pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)\n                vectors.append(torch.nn.functional.normalize(pooled, p=2, dim=1))\n        elapsed = time.perf_counter() - started\n        embeddings = torch.cat(vectors)\n        scores = embeddings[:len(requirements)] @ embeddings[len(requirements):].T\n        if not torch.isfinite(scores).all():\n            raise ValueError(\'Nonfinite model scores\')\n        rankings = []\n        for row, requirement in zip(scores.tolist(), requirements):\n            ranked = sorted([dict(e, score=s) for e, s in zip(evidence, row)],\n                            key=lambda e: (-e[\'score\'], e[\'id\']))\n            rankings.append({\'requirement\': requirement, \'ranked_evidence\': ranked})\n        result[\'arms\'][arm] = {\n            \'model_files_sha256\': {p.name: digest(p) for p in weights},\n            \'encoding_seconds\': elapsed,\n            \'token_lengths\': token_lengths,\n            \'truncated_text_count\': sum(n > 256 for n in token_lengths),\n            \'rankings\': rankings,\n        }\n        del model, tokenizer, embeddings, vectors\n    args.output.mkdir(parents=True, exist_ok=False)\n    (args.output / \'predictions.json\').write_text(\n        json.dumps(result, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    lines = [\'# AI1 — CV retrieval demo\', \'\',\n             \'Exploratory single-CV comparison. No verified labels: no accuracy or winner claimed.\', \'\']\n    for i, requirement in enumerate(requirements):\n        lines.extend([f"## {requirement[\'id\']}: {requirement[\'text\']}", \'\'])\n        for arm in result[\'arms\']:\n            lines.extend([f\'### {arm}\', \'\'])\n            for rank, e in enumerate(result[\'arms\'][arm][\'rankings\'][i][\'ranked_evidence\'][:3], 1):\n                lines.append(f"{rank}. **{e[\'id\']}**, cosine {e[\'score\']:.6f}: {e[\'text\']}")\n            lines.append(\'\')\n    (args.output / \'REPORT.md\').write_text(\'\\n\'.join(lines), encoding=\'utf-8\')\n    print(f\'Results: {args.output}\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--baseline\', type=Path, required=True)\n    parser.add_argument(\'--arm-c\', type=Path, required=True)\n    parser.add_argument(\'--input\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    run(parser.parse_args())\n'

input_path = work / "input.json"
runner_path = work / "compare_ai1_cv_demo.py"
input_path.write_text(json.dumps(INPUT, ensure_ascii=False, indent=2), encoding="utf-8")
runner_path.write_text(RUNNER, encoding="utf-8")
print("Requirements:", len(INPUT["requirements"]), "Evidence:", len(INPUT["evidence"]))
for r in INPUT["requirements"]:
    print(r["id"], r["text"])


In [ ]:
# Chỉ inference CPU; không optimizer, không tải model từ Internet.
output_dir = work / 'results'
cmd = [sys.executable, '-I', str(runner_path),
       '--baseline', str(baseline), '--arm-c', str(arm_c),
       '--input', str(input_path), '--output', str(output_dir)]
subprocess.run(cmd, check=True, cwd='/kaggle/working')


In [ ]:
from IPython.display import Markdown, display, FileLink
import zipfile

display(Markdown((output_dir / 'REPORT.md').read_text(encoding='utf-8')))

# ZIP chỉ chứa kết quả + đầu vào + script, không sao chép weights.
zip_path = work / 'ai1_cv_demo_results.zip'
with zipfile.ZipFile(zip_path, 'x', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [input_path, runner_path, *sorted(output_dir.iterdir())]:
        if p.is_file():
            zf.write(p, arcname=p.relative_to(work).as_posix())
print('Kết quả RIÊNG TƯ:', zip_path)
display(FileLink(str(zip_path)))
print('Nếu link không hoạt động: refresh Output, mở thư mục chạy in ở trên và tải ZIP.')
